In [2]:
from PIL import Image
import os

dataset_path = "dataset"

bad_files = 0

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        file_path = os.path.join(root, file)
        
        try:
            img = Image.open(file_path)
            img.verify()  # check if valid
        except:
            print(f"❌ Removing corrupted file: {file_path}")
            os.remove(file_path)
            bad_files += 1

print(f"\n✅ Cleaning done! Removed {bad_files} bad images.")

❌ Removing corrupted file: dataset\hazardous\img_100.jpg
❌ Removing corrupted file: dataset\hazardous\img_111.jpg
❌ Removing corrupted file: dataset\hazardous\img_112.jpg
❌ Removing corrupted file: dataset\hazardous\img_121.jpg
❌ Removing corrupted file: dataset\hazardous\img_122.jpg
❌ Removing corrupted file: dataset\hazardous\img_129.jpg
❌ Removing corrupted file: dataset\hazardous\img_135.jpg
❌ Removing corrupted file: dataset\hazardous\img_136.jpg
❌ Removing corrupted file: dataset\hazardous\img_138.jpg
❌ Removing corrupted file: dataset\hazardous\img_142.jpg
❌ Removing corrupted file: dataset\hazardous\img_151.jpg
❌ Removing corrupted file: dataset\hazardous\img_160.jpg
❌ Removing corrupted file: dataset\hazardous\img_161.jpg
❌ Removing corrupted file: dataset\hazardous\img_41.jpg
❌ Removing corrupted file: dataset\hazardous\img_55.jpg
❌ Removing corrupted file: dataset\hazardous\img_56.jpg
❌ Removing corrupted file: dataset\hazardous\img_57.jpg
❌ Removing corrupted file: dataset\

In [3]:
# =====================
# IMPORTS
# =====================
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# =====================
# CONFIG
# =====================
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_STAGE1 = 10
EPOCHS_STAGE2 = 5
DATASET_PATH = "dataset"

# =====================
# DATA GENERATOR
# =====================
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# =====================
# CLASS WEIGHTS
# =====================
labels = train_data.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weights = dict(enumerate(class_weights))

# =====================
# BASE MODEL
# =====================
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False  # freeze initially

# =====================
# CUSTOM HEAD
# =====================
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(train_data.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

# =====================
# STAGE 1 TRAINING
# =====================
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n🚀 Stage 1: Training top layers...")
model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS_STAGE1,
    class_weight=class_weights
)

# =====================
# STAGE 2: FINE-TUNING
# =====================
print("\n🔥 Stage 2: Fine-tuning...")

base_model.trainable = True

# Freeze most layers, unfreeze last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # VERY IMPORTANT
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS_STAGE2,
    class_weight=class_weights
)

# =====================
# SAVE MODEL
# =====================
model.save("waste_classifier.keras")

print("✅ Final fine-tuned model saved!")

Found 1502 images belonging to 7 classes.
Found 374 images belonging to 7 classes.

🚀 Stage 1: Training top layers...


c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 54s 1s/step - accuracy: 0.5007 - loss: 1.3550 - val_accuracy: 0.6898 - val_loss: 0.8869
Epoch 2/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.7210 - loss: 0.7995 - val_accuracy: 0.7273 - val_loss: 0.6961
Epoch 3/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.7650 - loss: 0.6592 - val_accuracy: 0.7727 - val_loss: 0.6614
Epoch 4/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 46s 981ms/step - accuracy: 0.7996 - loss: 0.5458 - val_accuracy: 0.7754 - val_loss: 0.6535
Epoch 5/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 47s 995ms/step - accuracy: 0.8236 - loss: 0.4822 - val_accuracy: 0.7594 - val_loss: 0.6782
Epoch 6/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 46s 983ms/step - accuracy: 0.8422 - loss: 0.4428 - val_accuracy: 0.7834 - val_loss: 0.6137
Epoch 7/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 46s 981ms/step - accuracy: 0.8515 - loss: 0.3926 - val_accuracy: 0.7861 - val_loss: 0.6609
Epoch 8/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 47s 992ms/step - accuracy: 0.8615 - loss: 0.3973 - val_accuracy: 0.8